# Mercado laboral en Canadá: paso 1, qué tablas existen y qué traen

**Por qué cambió:** las tablas completas pesan millones de filas (la de desempleo por provincia tiene 5,5 millones).
Bajarlas solo para mirarlas es lento y no sirve. Acá usamos la **API de metadatos** de StatCan: no descarga datos,
solo pide el catálogo y la ficha de cada tabla. Tarda segundos.

**Cómo usarlo:** *Entorno de ejecución > Ejecutar todo*. Son dos celdas.
1. Busca en el catálogo tablas sobre vacantes, inmigrantes y seguro de empleo.
2. Muestra título, fechas y categorías exactas de las 4 tablas que veníamos usando.

Copiá todo lo que imprimen ambas celdas y pegalo en el chat.

In [ ]:
import requests, pandas as pd

BASE = "https://www150.statcan.gc.ca/t1/wds/rest"

def api(metodo, payload=None):
    """Llama a la API de metadatos de StatCan. Si algo viene raro, lo muestra en vez de fallar en silencio."""
    if payload is None:
        r = requests.get(f"{BASE}/{metodo}", timeout=60)
    else:
        r = requests.post(f"{BASE}/{metodo}", json=payload, timeout=60)
    r.raise_for_status()
    return r.json()

catalogo = pd.DataFrame(api("getAllCubesListLite"))
print(f"Catalogo de StatCan: {len(catalogo):,} tablas")
print("Columnas disponibles:", list(catalogo.columns))
catalogo["productId"] = catalogo["productId"].astype(str)
catalogo["titulo"] = catalogo["cubeTitleEn"]

def buscar(*palabras, solo_trabajo=True, max_filas=25):
    """Tablas cuyo titulo contiene TODAS las palabras. Por defecto solo la serie 14-10 (mercado laboral)."""
    m = catalogo
    if solo_trabajo:
        m = m[m["productId"].str.startswith("1410")]
    for p in palabras:
        m = m[m["titulo"].str.contains(p, case=False, na=False)]
    cols = [c for c in ["productId", "titulo", "cubeStartDate", "cubeEndDate", "frequencyCode"] if c in m.columns]
    return m[cols].head(max_filas).to_string(index=False)

for etiqueta, palabras in {
    "VACANTES": ("vacanc",),
    "INMIGRANTES": ("immigrant",),
    "SEGURO DE EMPLEO": ("employment insurance",),
    "DESEMPLEO + provincia": ("unemployment", "province"),
}.items():
    print("\n" + "=" * 90 + f"\n{etiqueta}\n" + buscar(*palabras))

## Ficha de cada tabla

In [ ]:
PIDS = ["14100287", "14100011", "14100471", "14100443"]   # las 4 que veniamos usando

def esquema(pid, max_miembros=15):
    obj = api("getCubeMetadata", [{"productId": int(pid)}])[0]["object"]
    print("=" * 90)
    print(f"{pid}  {obj.get('cubeTitleEn')}")
    print(f"  Periodo: {obj.get('cubeStartDate')} a {obj.get('cubeEndDate')} | ultima carga: {obj.get('releaseTime')}")
    for d in obj.get("dimension", []):
        miembros = [m["memberNameEn"] for m in d.get("member", [])]
        print(f"\n  DIMENSION '{d.get('dimensionNameEn')}': {len(miembros)} categorias")
        for m in miembros[:max_miembros]:
            print(f"     - {m}")
        if len(miembros) > max_miembros:
            print(f"     ... y {len(miembros) - max_miembros} mas")

for pid in PIDS:
    try:
        esquema(pid)
    except Exception as e:
        print("=" * 90); print(f"{pid}: no pude leer ({type(e).__name__}: {e})")